# 5-Fold RealMLP_TD (use_ls=False, không dùng Target Encoding)

**Cấu hình chạy trên Kaggle:** Accelerator = GPU T4 x2, Internet = On (cài `pytabkit`).
- Tắt label smoothing mặc định (`use_ls=False`) và chọn `val_metric_name='1-auc_ovr'` phù hợp cho ROC-AUC.
- Học trực tiếp trên đặc trưng số, chữ số và biến phân loại (không dùng bảng Target Encoding của nhánh cây).
- Kết quả xuất ra `/kaggle/working/`: `submission.csv`, `oof_kaggle_realmlp_clean_5fold.csv`, `sub_kaggle_realmlp_clean_details.csv`.


In [ ]:
!pip install -q pytabkit
import torch, pytabkit
print('pytabkit:', getattr(pytabkit, '__version__', 'installed'), '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
"""Huấn luyện 5-Fold RealMLP_TD (use_ls=False, không dùng Target Encoding) trên Kaggle GPU."""

from __future__ import annotations

import gc
import os
from pathlib import Path
import time

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch

try:
    from pytabkit import RealMLP_TD_Classifier
    HAS_PYTABKIT = True
except Exception:
    RealMLP_TD_Classifier = None
    HAS_PYTABKIT = False

DRY_RUN = os.environ.get("KAGGLE_DRY_RUN", "0") == "1"
SEED = 42
N_SPLITS = 2 if DRY_RUN else 5
N_EPOCHS = 2 if DRY_RUN else 12
BATCH_SIZE = 256 if DRY_RUN else 512

ID_COLUMN = "id"
TARGET_COLUMN = "is_fraud"
NUMERIC_COLUMNS = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
]
FLOAT_COLUMNS = [
    "amount_usd",
    "account_balance_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "velocity_score",
    "merchant_risk_score",
]
BINARY_COLUMNS = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]
CATEGORICAL_COLUMNS = [
    "merchant_category",
    "card_type",
    "auth_method",
    "channel",
    "device_type",
]


def locate_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/playground-series-s6e11"),
        Path("/kaggle/input/competitions/playground-series-s6e11"),
        Path("dataset/playground-series-s6e11"),
        Path("../dataset/playground-series-s6e11"),
    ]
    for p in candidates:
        if (p / "train.csv").exists():
            return p
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for p in kaggle_root.rglob("train.csv"):
            if (p.parent / "test.csv").exists():
                return p.parent
    raise FileNotFoundError("Could not locate train.csv / test.csv.")


def locate_output_dir() -> Path:
    custom = os.environ.get("KAGGLE_OUT_DIR")
    if custom:
        out = Path(custom)
        out.mkdir(parents=True, exist_ok=True)
        return out
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    out = Path("artifacts/kaggle_realmlp_clean")
    out.mkdir(parents=True, exist_ok=True)
    return out


def safe_logit(p: np.ndarray | float) -> np.ndarray | float:
    return logit(np.clip(p, 1e-6, 1.0 - 1e-6))


def _leading_digits(int_arr: np.ndarray, k: int) -> np.ndarray:
    vals = np.maximum(int_arr, 0).astype(np.int64)
    out = np.zeros_like(vals)
    pos = vals > 0
    if not pos.any():
        return out
    v = vals[pos]
    digits = np.floor(np.log10(v.astype(np.float64))).astype(np.int64) + 1
    shift = np.maximum(digits - k, 0)
    pow10 = np.power(10, np.minimum(shift, 12), dtype=np.int64)
    out[pos] = v // pow10
    return out


def build_clean_no_te_tables(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Tạo 65 cột số/chữ số và 30 cột phân loại cho RealMLP (không dùng Target Encoding)."""
    frames = []
    for df in (train_df, test_df):
        d: dict[str, np.ndarray | pd.Series] = {}
        for c in NUMERIC_COLUMNS:
            d[c] = df[c].to_numpy(dtype=np.float32)
        for c in BINARY_COLUMNS:
            d[c] = df[c].to_numpy(dtype=np.float32)

        amt = df["amount_usd"].to_numpy(dtype=np.float64)
        bal = df["account_balance_usd"].to_numpy(dtype=np.float64)
        hrs = df["hours_since_last_txn"].to_numpy(dtype=np.float64)
        dist = df["distance_from_home_km"].to_numpy(dtype=np.float64)
        hour = df["time_of_day_hour"].to_numpy(dtype=np.float64)
        dow = df["day_of_week"].to_numpy(dtype=np.float64)

        d["flag_sum"] = df[BINARY_COLUMNS].sum(axis=1).to_numpy(dtype=np.float32)
        d["log_amt"] = np.log1p(np.maximum(amt, 0.0)).astype(np.float32)
        d["log_bal"] = np.log1p(np.maximum(bal, 0.0)).astype(np.float32)
        d["log_hrs"] = np.log1p(np.maximum(hrs, 0.0)).astype(np.float32)
        d["log_dist"] = np.log1p(np.maximum(dist, 0.0)).astype(np.float32)
        d["amt_over_bal"] = (amt / (np.maximum(bal, 0.0) + 1.0)).astype(np.float32)
        d["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0).astype(np.float32)
        d["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0).astype(np.float32)
        d["dow_sin"] = np.sin(2.0 * np.pi * dow / 7.0).astype(np.float32)
        d["dow_cos"] = np.cos(2.0 * np.pi * dow / 7.0).astype(np.float32)

        # Chữ số phần lẻ dạng số nguyên thô
        for c in FLOAT_COLUMNS:
            c100 = np.rint(df[c].to_numpy(dtype=np.float64) * 100.0).astype(np.int64)
            ip = np.floor_divide(c100, 100)
            d[f"{c}__dec2_num"] = np.mod(c100, 100).astype(np.float32)
            d[f"{c}__cent1_num"] = np.mod(c100, 10).astype(np.float32)
            d[f"{c}__cent2_num"] = np.mod(np.floor_divide(c100, 10), 10).astype(np.float32)
            d[f"{c}__unit_num"] = np.mod(ip, 10).astype(np.float32)
            d[f"{c}__tens_num"] = np.mod(np.floor_divide(ip, 10), 10).astype(np.float32)
            d[f"{c}__lead1_num"] = _leading_digits(ip, 1).astype(np.float32)

        # 30 cột phân loại (5 cột gốc, 13 giá trị số lặp lại và 12 khóa phần thập phân)
        cat_cols: list[str] = []
        for c in CATEGORICAL_COLUMNS:
            d[c] = df[c].astype(str)
            cat_cols.append(c)
        for c in NUMERIC_COLUMNS:
            c100 = np.rint(df[c].to_numpy(dtype=np.float64) * 100.0).astype(np.int64)
            cn = f"cat_ex__{c}"
            d[cn] = c100
            cat_cols.append(cn)
        for c in FLOAT_COLUMNS:
            c100 = np.rint(df[c].to_numpy(dtype=np.float64) * 100.0).astype(np.int64)
            for suf, arr in [
                ("dec2", np.mod(c100, 100)),
                ("cent2", np.mod(np.floor_divide(c100, 10), 10)),
            ]:
                cn = f"cat_{suf}__{c}"
                d[cn] = arr
                cat_cols.append(cn)

        frames.append(pd.DataFrame(d))
    return frames[0], frames[1], cat_cols


def encode_fold_categories(
    X_tr: pd.DataFrame,
    X_va: pd.DataFrame,
    X_te: pd.DataFrame,
    cat_cols: list[str],
    min_count: int = 2,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Mã hóa cột phân loại theo từng fold: gán giá trị hiếm hoặc chưa gặp về chỉ số 0."""
    tr_out, va_out, te_out = X_tr.copy(), X_va.copy(), X_te.copy()
    for c in cat_cols:
        if c in CATEGORICAL_COLUMNS:
            vals = sorted(tr_out[c].unique().tolist())
            mp = {v: i + 1 for i, v in enumerate(vals)}
            tr_codes = tr_out[c].map(mp).fillna(0).to_numpy(dtype=np.int32)
            va_codes = va_out[c].map(mp).fillna(0).to_numpy(dtype=np.int32)
            te_codes = te_out[c].map(mp).fillna(0).to_numpy(dtype=np.int32)
            n_cat = len(vals) + 1
        else:
            tr_v = tr_out[c].to_numpy(dtype=np.int64)
            va_v = va_out[c].to_numpy(dtype=np.int64)
            te_v = te_out[c].to_numpy(dtype=np.int64)
            uniq, cnt = np.unique(tr_v, return_counts=True)
            keep = uniq[cnt >= min_count]

            def _map_int(arr: np.ndarray) -> np.ndarray:
                if keep.size == 0:
                    return np.zeros(len(arr), dtype=np.int32)
                idx = np.searchsorted(keep, arr)
                idx_s = np.minimum(idx, keep.size - 1)
                return np.where(keep[idx_s] == arr, idx_s + 1, 0).astype(np.int32)

            tr_codes, va_codes, te_codes = _map_int(tr_v), _map_int(va_v), _map_int(te_v)
            n_cat = int(keep.size) + 1

        cats = np.arange(n_cat, dtype=np.int32)
        tr_out[c] = pd.Categorical(tr_codes, categories=cats)
        va_out[c] = pd.Categorical(va_codes, categories=cats)
        te_out[c] = pd.Categorical(te_codes, categories=cats)
    return tr_out, va_out, te_out


def fit_realmlp_no_ls(
    X_tr: pd.DataFrame, y_tr: np.ndarray,
    X_va: pd.DataFrame, y_va: np.ndarray,
    X_te: pd.DataFrame, cat_cols: list[str],
    device: str, seed: int, fold_id: int,
) -> tuple[np.ndarray, np.ndarray]:
    if not HAS_PYTABKIT or RealMLP_TD_Classifier is None:
        raise RuntimeError("pytabkit is required! Make sure Internet is ON and Cell 1 installed pytabkit.")

    t0 = time.perf_counter()
    # Ưu tiên thước đo AUC khi dừng sớm, dự phòng cross_entropy nếu phiên bản khác (luôn giữ use_ls=False)
    for metric_name in ("1-auc_ovr", "cross_entropy"):
        try:
            model = RealMLP_TD_Classifier(
                device=device,
                random_state=seed,
                n_cv=1,
                n_refit=0,
                n_epochs=N_EPOCHS,
                batch_size=BATCH_SIZE,
                use_ls=False,
                val_metric_name=metric_name,
                verbosity=1,
            )
            model.fit(X_tr, y_tr, X_val=X_va, y_val=y_va, cat_col_names=cat_cols)
            va_p = model.predict_proba(X_va)[:, 1]
            te_p = model.predict_proba(X_te)[:, 1]
            va_l = safe_logit(va_p).astype(np.float32)
            te_l = safe_logit(te_p).astype(np.float32)
            auc = float(roc_auc_score(y_va, va_l))
            print(
                f"  --> [RealMLP-Clean F{fold_id+1} metric={metric_name}] Val AUC: {auc:.6f}"
                f" | logit mean={va_l.mean():+.2f} std={va_l.std():.2f} | {time.perf_counter()-t0:.1f}s",
                flush=True,
            )
            del model
            if device == "cuda":
                torch.cuda.empty_cache()
            return va_l, te_l
        except Exception as exc:
            if metric_name == "cross_entropy":
                raise
            print(f"    metric={metric_name} raised {type(exc).__name__}: {exc} -> retrying with cross_entropy (use_ls=False)...", flush=True)
    raise RuntimeError("Unreachable")


def main():
    t_start = time.perf_counter()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data_dir = locate_data_dir()
    out_dir = locate_output_dir()
    print(f"Device: {device} | pytabkit={HAS_PYTABKIT} | use_ls=False | epochs={N_EPOCHS} | bs={BATCH_SIZE}", flush=True)

    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")
    sample_sub = pd.read_csv(data_dir / "sample_submission.csv", usecols=[ID_COLUMN])

    if DRY_RUN:
        train_df = pd.concat([train_df[train_df[TARGET_COLUMN] == 1].head(200), train_df[train_df[TARGET_COLUMN] == 0].head(1800)], ignore_index=True)
        test_df = test_df.head(400).reset_index(drop=True)
        sample_sub = pd.DataFrame({ID_COLUMN: test_df[ID_COLUMN]})

    y_all = train_df[TARGET_COLUMN].to_numpy(dtype=np.int64)
    train_ids = train_df[ID_COLUMN].to_numpy()
    test_ids = test_df[ID_COLUMN].to_numpy()
    sample_ids = pd.Index(sample_sub[ID_COLUMN])

    X_tr_all, X_te_all, cat_cols = build_clean_no_te_tables(train_df, test_df)
    print(f"Built clean NO-TE table: {X_tr_all.shape[1]} cols ({len(cat_cols)} categorical, {X_tr_all.shape[1]-len(cat_cols)} numeric)", flush=True)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_l = np.zeros(len(train_df), dtype=np.float64)
    test_l = np.zeros(len(test_df), dtype=np.float64)
    fold_aucs = []

    for fold_id, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
        print(f"\n{'=' * 72}\nFOLD {fold_id + 1}/{N_SPLITS} (Train={len(tr_idx):,} | Val={len(va_idx):,})\n{'=' * 72}", flush=True)
        X_tr_c, X_va_c, X_te_c = encode_fold_categories(
            X_tr_all.iloc[tr_idx].reset_index(drop=True),
            X_tr_all.iloc[va_idx].reset_index(drop=True),
            X_te_all,
            cat_cols,
            min_count=2,
        )
        va_pred, te_pred = fit_realmlp_no_ls(
            X_tr_c, y_all[tr_idx], X_va_c, y_all[va_idx], X_te_c,
            cat_cols, device=device, seed=SEED + fold_id, fold_id=fold_id,
        )
        oof_l[va_idx] = va_pred
        test_l += te_pred / N_SPLITS
        fold_aucs.append(float(roc_auc_score(y_all[va_idx], va_pred)))
        del X_tr_c, X_va_c, X_te_c
        gc.collect()

    g_auc = float(roc_auc_score(y_all, oof_l))
    rec05 = float(recall_score(y_all, expit(oof_l) >= 0.5, zero_division=0))
    print(f"\n{'=' * 72}")
    print(f"RealMLP-TD (use_ls=False, NO Tree TE) {N_SPLITS}-Fold Global OOF AUC : {g_auc:.6f}")
    print(f"Fold Mean ± Std: {np.mean(fold_aucs):.6f} ± {np.std(fold_aucs):.6f} | Recall@0.5: {rec05:.6f}")
    print(f"Total elapsed: {(time.perf_counter() - t_start) / 60:.1f} min")
    print(f"{'=' * 72}", flush=True)

    pd.DataFrame({
        ID_COLUMN: train_ids,
        TARGET_COLUMN: y_all,
        "pred_prob": expit(oof_l),
        "logit_realmlp": oof_l,
    }).to_csv(out_dir / "oof_kaggle_realmlp_clean_5fold.csv", index=False)

    sub_df = pd.DataFrame({ID_COLUMN: sample_ids})
    sub_df[TARGET_COLUMN] = pd.Series(expit(test_l), index=test_ids).reindex(sample_ids).to_numpy()
    sub_df.to_csv(out_dir / "submission.csv", index=False)
    sub_df.to_csv(out_dir / "sub_kaggle_realmlp_clean_5fold.csv", index=False)

    det_df = pd.DataFrame({ID_COLUMN: sample_ids})
    det_df["logit_realmlp"] = pd.Series(test_l, index=test_ids).reindex(sample_ids).to_numpy()
    det_df.to_csv(out_dir / "sub_kaggle_realmlp_clean_details.csv", index=False)
    print(f"Saved oof_kaggle_realmlp_clean_5fold.csv, sub_kaggle_realmlp_clean_details.csv, submission.csv in {out_dir}", flush=True)


In [ ]:
main()
